# 06 · Full transcript collection, and what full coverage buys

`04` collected a stratified sample of 600 videos and resolved 763 transcripts in total. That was
enough to establish that transcripts are collectable and that one finding — TechLinked's shift
toward AI coverage — survives every parameter choice. It is not enough to answer the question this
notebook is for.

**The question.** Restricted to what the company published *after* the decline, which subjects
actually perform, and does anything about that connect to what the storefront sells?

That question is per-channel and per-subject at once, and the sample does not support it. Post-decline
Techquickie has 82 transcripts out of 142 videos; split those across a dozen subjects and most cells
hold single digits. Full coverage removes the problem rather than working around it.

**What this notebook does**

1. Builds the remaining fetch queue and reports exactly how much is missing.
2. Runs a slow, resumable, single-threaded collection designed to be left overnight and
   interrupted at any point without losing work.
3. Runs the post-decline subject analysis. **The analysis code does not care how much has been
   collected** — it runs tonight against whatever is on disk and again tomorrow against the full
   corpus, unchanged.

**What it deliberately does not do.** No conversion, no revenue, no attribution from a video to a
sale. The storefront endpoint carries prices and stock status and no units sold. Everything in
section 3 is about *exposure* and is written as future work, not as a result.

## 0. Environment

In [ ]:
%pip install -q yt-dlp curl-cffi pandas numpy scikit-learn matplotlib

import sys, shutil
print("Kernel:", sys.executable)
print("yt-dlp:", shutil.which("yt-dlp") or "NOT ON PATH")
for mod in ["pandas", "numpy", "sklearn", "matplotlib", "curl_cffi"]:
    try:
        __import__(mod); print(f"  ok   {mod}")
    except ImportError as e:
        print(f"  FAIL {mod} -> {e}")

## 1. Scope and what is missing

The target set is the same one `04` defined, and it is deliberately not the whole catalogue: the
four channels, no WAN Show, 2021-01 through 2026-06, split at 2023-07-01. Going back further would
add years of low-quality auto-captions that no part of the analysis uses.

The cache is append-only and shared with `04`. Two rules matter and both have already caused a
mistake in this project:

- **Refusals are never written to disk.** A `429` is information about the connection, not about
  the video, so those IDs stay eligible. Writing them once made coverage read 27% when it was 99.7%.
- **De-duplication keeps the *last* record**, because a video refused early and retrieved later
  appears twice.

In [ ]:
import os, json, time, random, pathlib, re, subprocess, shutil, tempfile
import pandas as pd, numpy as np

CWD  = pathlib.Path.cwd()
NB   = next((p for p in [CWD, CWD/"notebook", *CWD.parents] if (p/"data").is_dir()), CWD)
DATA = NB/"data"; RAW = DATA/"content"; RAW.mkdir(exist_ok=True)
FIG  = NB/"figures"; FIG.mkdir(exist_ok=True)
TR_CACHE = RAW/"transcripts.jsonl"

WINDOWS  = {"pre": ("2021-01-01", "2023-06-30"), "post": ("2023-07-01", "2026-06-30")}
CHANNELS = ["LinusTechTips", "Techquickie", "ShortCircuit", "TechLinked"]

vids = pd.read_csv(DATA/"videos_clean.csv", parse_dates=["date"])
sel  = vids[vids.channel.isin(CHANNELS) & (vids.format != "wan_show")].copy()
sel["window"] = np.select(
    [sel.date.between(*WINDOWS["pre"]), sel.date.between(*WINDOWS["post"])],
    ["pre", "post"], default=None)
target = sel[sel.window.notna()].reset_index(drop=True)


def read_cache(path=TR_CACHE):
    if not path.exists():
        return pd.DataFrame(columns=["id", "transcript", "status"])
    return (pd.DataFrame([json.loads(l) for l in path.open() if l.strip()])
              .drop_duplicates("id", keep="last"))


tr = read_cache()
# Resolved = we know the answer and asking again would tell us nothing new.
RESOLVED = {"ok", "no_captions", "too_short"}
have = set(tr[tr.status.isin(RESOLVED)].id) if len(tr) else set()
target["have"] = target.id.isin(have)

print(f"Target set: {len(target):,} videos\n")
print(target.pivot_table(index="channel", columns="window", values="id",
                         aggfunc="count", fill_value=0).to_string())
print(f"\nResolved so far: {target.have.sum():,} ({target.have.mean():.0%})")
print(f"Remaining:       {(~target.have).sum():,}\n")
print(target[~target.have].groupby(["window", "channel"]).size()
            .unstack(fill_value=0).to_string())

### Queue order

The queue is stratified by channel and window and then **interleaved**, so the twelve strata drain
at the same rate. This is the same reasoning as the sampling in `04`: if the run is interrupted at
3am, or the address is refused after 2,000 requests, what is on disk is still balanced across the
comparison that matters. A queue in publication order would leave the analysis with all of one
channel and none of another.

In [ ]:
rng = np.random.default_rng(0)

strata = [g.sample(frac=1, random_state=0).id.tolist()
          for _, g in target[~target.have].groupby(["channel", "window"])]

# Round-robin across strata: take one from each in turn until all are empty.
todo = [s.pop(0) for _ in range(max((len(s) for s in strata), default=0))
        for s in strata if s]

print(f"{len(todo):,} videos queued")
if todo:
    head = target.set_index("id").loc[todo[:12]]
    print("\nFirst twelve, showing the interleave:")
    print(head[["channel", "window", "date"]].to_string())

## 2. The collection run

Written for an unattended overnight run, which changes the design in three ways compared to `04`:

**It survives a refusal instead of stopping at one.** `04` stopped after five consecutive
failures, which is right for a 90-minute supervised run and wrong here — a transient rate limit at
2am would waste the rest of the night. This backs off exponentially (1, 2, 4 … up to 15 minutes),
returns the refused ID to the back of the queue, and only gives up after `HARD_STOP` consecutive
failures, which indicates the address is actually blocked rather than throttled.

**It writes one line per success and flushes immediately.** Killing the kernel, closing the laptop,
or hitting interrupt costs at most the video in flight. Re-running the cell picks up where it left
off, because the queue is rebuilt from what is on disk.

**It reports an ETA**, so the decision to leave it running is an informed one.

`RUN = False` is the default so that a stray *Run All* can never start a six-hour scrape.
**Set it to `True` in the cell below to begin.**

In [ ]:
RUN         = False        # <-- set True to start the overnight collection
MAX_HOURS   = 8
DELAY       = (1.5, 4.0)   # random pause between requests, seconds
HARD_STOP   = 12           # consecutive failures before concluding the address is blocked
IMPERSONATE = True         # needs curl-cffi

if not shutil.which("yt-dlp"):
    print("yt-dlp is not on PATH — run:  %pip install -q yt-dlp curl-cffi")
    print("Collection disabled until it is.")
    RUN = False
scratch = pathlib.Path(tempfile.mkdtemp())


def json3_to_text(p):
    d = json.loads(pathlib.Path(p).read_text())
    return " ".join(s.get("utf8", "") for e in d.get("events", []) if "segs" in e
                    for s in e["segs"]).replace("\n", " ").strip()


def fetch_one(vid, timeout=90):
    """Returns (text, status). Refusal and absence are different outcomes and are kept
    separate — only one of them is information about the video."""
    cmd = ["yt-dlp", "--skip-download", "--write-auto-sub", "--write-sub",
           "--sub-lang", "en.*", "--sub-format", "json3", "--no-warnings",
           "-o", str(scratch/"%(id)s"), f"https://youtu.be/{vid}"]
    if IMPERSONATE:
        cmd[1:1] = ["--impersonate", "chrome"]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return None, "timeout"
    files = sorted(scratch.glob(f"{vid}*.json3"))
    if files:
        text = json3_to_text(files[0])
        for f in files:
            f.unlink()
        return (text, "ok") if len(text) > 200 else (text, "too_short")
    err = (r.stderr or "") + (r.stdout or "")
    if "429" in err or "Too Many Requests" in err:
        return None, "blocked"
    if "no subtitles" in err.lower() or "no automatic captions" in err.lower():
        return None, "no_captions"
    return None, "error"


print("RUN =", RUN, "· queue:", f"{len(todo):,}")

### Probe first

Three requests before committing the night. The most informative thing learned during `04` was
that a refusal is a property of the *connection*, not of the code — the first attempt was refused
after 160 requests and the identical code succeeded on a phone hotspot. If the probe comes back
refused, change networks; do not change the code.

In [ ]:
if RUN and todo:
    probe = [fetch_one(v)[1] for v in todo[:3]]
    print("Probe:", probe)
    if probe.count("blocked") >= 2:
        print("\nRefused on this connection. Switch networks and re-run — do not touch the code.")
        RUN = False
    else:
        todo = todo[3:]       # the probe resolved these; don't pay for them twice
else:
    print("RUN is False — no requests made.")

In [ ]:
if RUN and todo:
    t0 = time.time()
    queue = list(todo)
    stats, streak, done, ok_n = {}, 0, 0, 0
    backoff = 60

    try:
        with TR_CACHE.open("a") as f:
            while queue:
                elapsed_h = (time.time() - t0) / 3600
                if elapsed_h >= MAX_HOURS:
                    print(f"\nReached the {MAX_HOURS}-hour budget with {len(queue):,} left.")
                    break

                vid = queue.pop(0)
                time.sleep(random.uniform(*DELAY))
                text, status = fetch_one(vid)
                stats[status] = stats.get(status, 0) + 1

                if status in ("blocked", "timeout", "error"):
                    streak += 1
                    if streak >= HARD_STOP:
                        print(f"\n{HARD_STOP} consecutive failures — the address looks blocked "
                              f"rather than throttled. Stopping with {len(queue):,} left.")
                        break
                    if status == "blocked":
                        queue.append(vid)          # not the video's fault; try it again later
                        wait = min(backoff * 2 ** (streak - 1), 900)
                        print(f"  refused ({streak}/{HARD_STOP}) — backing off {wait//60}m{wait%60:02d}s")
                        time.sleep(wait)
                    continue

                streak = 0
                # Only resolved outcomes are persisted. Refusals stay off disk by design.
                f.write(json.dumps({"id": vid, "transcript": text, "status": status}) + "\n")
                f.flush()
                done += 1; ok_n += (status == "ok")

                if done % 50 == 0:
                    mins  = (time.time() - t0) / 60
                    rate  = done / max(mins, 1e-9)
                    eta_h = len(queue) / rate / 60 if rate else float("nan")
                    print(f"  {done:>5,} resolved · {ok_n:>5,} with text · {mins:>5.0f}m "
                          f"· {rate:4.1f}/min · ETA {eta_h:4.1f}h · {stats}")
    except KeyboardInterrupt:
        print(f"\nInterrupted. {done:,} resolved this run; everything is already on disk.")

    print(f"\nRun finished after {(time.time()-t0)/60:.0f} minutes · {done:,} resolved · {stats}")
elif RUN:
    print("Queue is empty — nothing to collect.")

### Progress

Safe to run at any time, including while the cell above is still going: it reads the cache from
disk rather than from memory. Use it to check on the run in the morning.

In [ ]:
tr = read_cache()
have = set(tr[tr.status.isin(RESOLVED)].id) if len(tr) else set()
got  = set(tr[tr.status == "ok"].id) if len(tr) else set()
target["have"] = target.id.isin(got)

print(tr.status.value_counts().to_string() if len(tr) else "cache empty")
print(f"\n{target.have.sum():,} of {len(target):,} target videos have a transcript "
      f"({target.have.mean():.0%})\n")
print(target.pivot_table(index="channel", columns="window", values="have",
                         aggfunc=["sum", "mean"]).round(2).to_string())

# Retrieval must not itself be selective — if what we failed to get skews high or low
# on views, every downstream comparison inherits that skew.
if target.have.sum() >= 40:
    a, b = target[target.have], target
    print(f"\nRetrieved vs full target set:")
    print(f"  median rbi    {a.rbi.median():.2f}  vs  {b.rbi.median():.2f}")
    print(f"  median views  {a.views.median():,.0f}  vs  {b.views.median():,.0f}")

## 3. What post-decline coverage is for

Everything below runs against whatever is on disk. It is written to be re-run after the collection
finishes; the numbers will change and the structure will not.

The restriction to post-decline output is the point. Comparing subjects across the whole catalogue
mixes eras, and view counts are cumulative, so an older video has had longer to accumulate them —
`02` established that the accrual curve cannot be recovered from public data. Staying inside one
era removes the confound instead of adjusting for it. The rolling baseline index compares each
video to the median of the nine that preceded it on the same channel and format, so it is already
local in time.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

tr_ok = read_cache(); tr_ok = tr_ok[tr_ok.status == "ok"][["id", "transcript"]]

# sort_values("id") is not cosmetic. KMeans++ initialisation depends on row order, so an
# unsorted merge gives a different partition on every run and the labels stop meaning
# anything. Fixing the order is what makes the rest of this section reproducible.
post = (target[target.window == "post"].merge(tr_ok, on="id")
        .sort_values("id").reset_index(drop=True))
post["chars"] = post.transcript.str.len()

print(f"{len(post):,} post-decline videos with a transcript "
      f"({len(post)/max((target.window=='post').sum(),1):.0%} of the post-decline set)\n")
print(post.groupby("channel").agg(n=("id","size"), med_min=("duration_min","median"),
                                  med_chars=("chars","median")).round(1).to_string())

FILLER = r"\b(uh|um|like|you know|i mean|kind of|sort of|gonna|wanna|okay|alright|yeah|so|right)\b"

def clean(t):
    t = re.sub(r"[^a-z0-9 ]", " ", str(t).lower())
    return re.sub(r"\s+", " ", re.sub(FILLER, " ", t)).strip()

STOP = ("the a an and or of to in on for with is are this that we i you my our it its at by from "
        "as be new video do does did can will just get got make made most best but not out up all "
        "more very vs your what why how much many when who has have had been was were there here "
        "now then also one two three into over about after before than them their these those go "
        "going want really actually think see look thing things guys today let lets need put take "
        "come said say little bit lot pretty good great big time way well because if then").split()

post["txt"] = post.transcript.map(clean)

# --- Channel-specific boilerplate -------------------------------------------------
# A first pass clustered on presenter script rather than subject: one cluster came back on
# "dislike, subscribe follow, forget subscribe" and had the highest relative performance in
# the set. Hand-picking stop words would let me tune the answer, so the rule is mechanical:
# drop a term when it appears in more than 60% of ONE channel's videos. That is what a fixed
# line is — something a channel says every episode.
cv = CountVectorizer(max_features=15000, ngram_range=(1, 2), min_df=8, max_df=.5,
                     binary=True, stop_words=STOP)
B = cv.fit_transform(post.txt); vocab = np.array(cv.get_feature_names_out())
mask = np.zeros(len(vocab), bool)
print("\nDropped as channel boilerplate (within-channel document frequency > 60%):\n")
for ch, idx in post.groupby("channel").groups.items():
    ii  = post.index.get_indexer(idx)
    df  = np.asarray(B[ii].mean(axis=0)).ravel()
    hit = df > .60; mask |= hit
    print(f"  {ch:15s} {hit.sum():>2}  {', '.join(vocab[hit][np.argsort(-df[hit])][:12])}")
BOILER = set(vocab[mask])
print(f"\n{len(BOILER)} terms removed.")

vec = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), min_df=8, max_df=.5,
                      sublinear_tf=True, stop_words=list(set(STOP) | BOILER))
X = vec.fit_transform(post.txt); terms = np.array(vec.get_feature_names_out())
print(f"{X.shape[0]:,} documents · {X.shape[1]:,} terms")

### Clusters are labelled by their own top terms, not by hand

The sampled version of this notebook asked for a hand-written `NAMES` dictionary. That turned out
to be unusable. Cluster IDs are arbitrary, and because KMeans++ depends on row order they changed
between two runs of the same code — a mapping written against one run silently mislabelled the
next, and in one case marked a news cluster as boilerplate.

So no cluster is named by hand anywhere below. Each is labelled by its own three highest-weighted
terms, which regenerates correctly whatever the partition turns out to be. It reads less neatly
and it cannot be quietly wrong.

In [ ]:
Z = TruncatedSVD(150, random_state=0).fit_transform(X)
Z = Z/np.clip(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9, None)
post["cluster"] = KMeans(14, n_init=10, random_state=0).fit_predict(Z)

label = {c: ", ".join(terms[np.asarray(X[(post.cluster == c).values].mean(axis=0))
                            .ravel().argsort()[::-1][:3]]) for c in range(14)}
post["topic"] = post.cluster.map(label)

g = post.groupby("topic")
perf = pd.DataFrame({
    "n": g.size(), "rbi": g.rbi.median(), "med_views": g.views.median(),
    "like_%":     g.apply(lambda d: 100*d.likes.sum()/d.views.sum(), include_groups=False),
    "cmt_per_1k": g.apply(lambda d: 1000*d.comments.sum()/d.views.sum(), include_groups=False),
}).round(2).sort_values("rbi", ascending=False)
print(perf.to_string())
perf.to_csv(DATA/"post_decline_topics.csv")

print(f"\nMedians span {perf.rbi.min():.2f}–{perf.rbi.max():.2f} across {len(perf)} subjects.")
print("Subject on its own explains very little of relative performance. That was already")
print("true on the sampled corpus and 3,400 more transcripts did not change it.")

### The question full coverage was actually for

The company-wide table is not worth an overnight run. This is.

At 17% coverage a subject split by channel left single-digit cells. At 99% it does not, and one
contrast is large: **ShortCircuit performs far below its own baseline on desktop build hardware —
power supplies, motherboards, fans — while Linus Tech Tips covering the same ground does not.**

A single clustering is not evidence of that, because the partition itself is a choice. So the
claim is tested the same way `05` tested the AI cluster: re-run across 24 configurations, and in
each one locate the cluster by its *content* rather than by its ID, using anchor terms. If the
result depends on the parameters, this will show it.

This cell takes two to three minutes.

In [ ]:
ANCHOR = {"power supply", "motherboard", "supply", "fans", "fan",
          "cooling", "cooler", "psu", "cpu cooler", "case"}
rows = []

for ncomp in [100, 150, 200]:
    Zi = TruncatedSVD(ncomp, random_state=0).fit_transform(X)
    Zi = Zi/np.clip(np.linalg.norm(Zi, axis=1, keepdims=True), 1e-9, None)
    for K in [10, 12, 14, 16]:
        for seed in [0, 1]:
            lab = KMeans(K, n_init=5, random_state=seed).fit_predict(Zi)
            best, score = None, 0
            for c in range(K):
                s = lab == c
                top = set(terms[np.asarray(X[s].mean(axis=0)).ravel().argsort()[::-1][:12]])
                if len(top & ANCHOR) > score:
                    best, score = c, len(top & ANCHOR)
            if best is None:
                continue
            s   = lab == best
            top = list(terms[np.asarray(X[s].mean(axis=0)).ravel().argsort()[::-1][:5]])
            a = post.loc[s & (post.channel == "ShortCircuit").values,  "rbi"].dropna()
            b = post.loc[s & (post.channel == "LinusTechTips").values, "rbi"].dropna()
            if len(a) < 10 or len(b) < 10:
                continue
            rows.append(dict(ncomp=ncomp, K=K, seed=seed, nSC=len(a), SC=a.median(),
                             nLTT=len(b), LTT=b.median(),
                             p=mannwhitneyu(a, b).pvalue, top=", ".join(top)))

stab = pd.DataFrame(rows)
# Did the anchor land on the intended cluster, or drag in a neighbouring one?
stab["on_target"] = stab.top.str.contains("power supply|motherboard|fans")
print(stab.round(4).to_string(index=False))
stab.to_csv(DATA/"topic_stability_post.csv", index=False)

on = stab[stab.on_target]
print(f"\n{len(stab)} configurations · anchor on target in {stab.on_target.sum()}")
print(f"ShortCircuit below Linus Tech Tips in {(stab.SC < stab.LTT).sum()} of {len(stab)}")
print(f"p < 0.05 in {(stab.p < .05).sum()} · p < 0.01 in {(stab.p < .01).sum()}")
print(f"On target: ShortCircuit {on.SC.min():.2f}–{on.SC.max():.2f}, "
      f"Linus Tech Tips {on.LTT.min():.2f}–{on.LTT.max():.2f}")

In [ ]:
def boot_ci(x, n=8000, seed=0):
    r = np.random.default_rng(seed); x = np.asarray(x)
    return np.percentile(np.median(r.choice(x, (n, len(x)), replace=True), axis=1), [2.5, 97.5])

fig, ax = plt.subplots(1, 2, figsize=(13.6, 5.0), gridspec_kw={"width_ratios": [1.25, 1]})

d = pd.DataFrame({"n": g.size(), "rbi": g.rbi.median()}).sort_values("rbi")
lo, hi = zip(*[boot_ci(post[post.topic == t].rbi.dropna()) for t in d.index])
y = np.arange(len(d))
ax[0].hlines(y, lo, hi, color="#9BbFdE", lw=3.2, zorder=2)
ax[0].plot(d.rbi, y, "o", color="#2E86DE", ms=6, zorder=3)
ax[0].axvline(1, color="#444", lw=1, ls="--")
ax[0].set_yticks(y); ax[0].set_yticklabels([f"{t}  (n={n})" for t, n in zip(d.index, d.n)], fontsize=8)
ax[0].set_xlabel("Rolling-baseline index  (1.0 = matched recent neighbours)")
ax[0].set_title("Company-wide, subject explains little", loc="left", weight="bold")
ax[0].set_ylim(-1.6, len(d)-.5)
ax[0].text(.98, .02, f"{len(d)} clusters, labelled by their own top terms rather than by hand.\n"
                     f"Medians span {d.rbi.min():.2f}–{d.rbi.max():.2f}; every interval touches 1.0.",
           transform=ax[0].transAxes, ha="right", va="bottom", fontsize=8.2, color="#555")

m = stab.on_target
ax[1].plot([.5, 1.3], [.5, 1.3], ls="--", color="#444", lw=1, zorder=1)
ax[1].scatter(stab.LTT[m], stab.SC[m], s=54, color="#2E86DE", zorder=3,
              label=f"anchor on the build-hardware cluster ({m.sum()})")
ax[1].scatter(stab.LTT[~m], stab.SC[~m], s=54, facecolor="none", edgecolor="#E74C3C", lw=1.6,
              zorder=3, label=f"anchor matched another cluster ({(~m).sum()})")
ax[1].set_xlabel("Linus Tech Tips — median rbi"); ax[1].set_ylabel("ShortCircuit — median rbi")
ax[1].set_xlim(.55, 1.28); ax[1].set_ylim(.55, 1.28)
ax[1].set_title(f"{len(stab)} configurations, {len(stab)} times the same direction",
                loc="left", weight="bold")
ax[1].legend(frameon=False, fontsize=8.2, loc="upper left")
ax[1].text(.035, .80, "3 component counts × 4 values of k × 2 seeds.\n"
                      "Every point sits below the diagonal; every p < 0.05.\n"
                      "Where the anchor lands on the intended cluster:\n"
                      f"ShortCircuit {on.SC.min():.2f}–{on.SC.max():.2f}, "
                      f"Linus Tech Tips {on.LTT.min():.2f}–{on.LTT.max():.2f}.",
           transform=ax[1].transAxes, ha="left", va="top", fontsize=8.0, color="#555")
for a in ax: a.spines[["top", "right"]].set_visible(False); a.grid(alpha=.22)
plt.tight_layout(); plt.savefig(FIG/"post_decline_topics.png", bbox_inches="tight"); plt.show()

In [ ]:
# ── for Remotion ──────────────────────────────────────────────────────────────
# Exports two things per figure: a transparent PLATE (axes, ticks, labels, titles,
# reference lines — no data marks) and a JSON of the series plus the geometry needed
# to map data coordinates onto the plate in pixels.
#
# The plate is rebuilt from scratch rather than stripped from the original, so a
# reference line can never be removed by accident. Numbers still come only from here.
REM = FIG/"remotion"; REM.mkdir(exist_ok=True)

def plate_geometry(fig, axes):
    """Pixel box of each axes plus its data limits — everything Remotion needs."""
    w_px, h_px = fig.get_size_inches()*fig.dpi
    out = []
    for a in axes:
        p = a.get_position()
        out.append({
            "px": {"x": p.x0*w_px, "y": (1-p.y1)*h_px, "w": p.width*w_px, "h": p.height*h_px},
            "xlim": list(a.get_xlim()), "ylim": list(a.get_ylim()),
            "xscale": a.get_xscale(), "yscale": a.get_yscale(),
        })
    return {"figure_px": {"w": w_px, "h": h_px}, "dpi": fig.dpi, "axes": out}

def save_plate(fig, name, series):
    fig.savefig(REM/f"{name}_plate.png", bbox_inches=None, transparent=True)
    geo = plate_geometry(fig, fig.axes)
    (REM/f"{name}.json").write_text(json.dumps({**geo, "series": series}, indent=1))
    print(f"  {name}_plate.png  +  {name}.json   "
          f"({geo['figure_px']['w']:.0f}x{geo['figure_px']['h']:.0f}px, {len(series)} series)")

# ---- post_decline_topics, left panel: dot-and-interval per subject ----------------
dd = pd.DataFrame({"n": g.size(), "rbi": g.rbi.median()}).sort_values("rbi")
lo_, hi_ = zip(*[boot_ci(post[post.topic == t].rbi.dropna()) for t in dd.index])
series = [{"name": t, "color": "#2E86DE", "kind": "interval", "y": float(i),
           "point": float(v), "lo": float(a), "hi": float(b), "label": f"{t}  (n={n})"}
          for i, (t, v, n, a, b) in enumerate(zip(dd.index, dd.rbi, dd.n, lo_, hi_))]
fig, ax = plt.subplots(figsize=(7.6, 5.0))
ax.axvline(1, color="#444", lw=1, ls="--")
yy = np.arange(len(dd))
ax.set_yticks(yy); ax.set_yticklabels([f"{t}  (n={n})" for t, n in zip(dd.index, dd.n)], fontsize=8)
ax.set_xlabel("Rolling-baseline index  (1.0 = matched recent neighbours)")
ax.set_title("Company-wide, subject explains little", loc="left", weight="bold")
ax.set_ylim(-1.6, len(dd)-.5); ax.set_xlim(min(lo_)-.03, max(hi_)+.03)
ax.spines[["top","right"]].set_visible(False); ax.grid(axis="x", alpha=.25)
plt.tight_layout()
save_plate(fig, "post_decline_topics_left", series); plt.close(fig)

> **The result.** Across 24 configurations — three component counts, four values of k, two
> seeds — ShortCircuit's median relative performance on desktop build hardware sits below Linus
> Tech Tips' **24 times out of 24**, and every configuration returns p < 0.05. In the 21 where the
> anchor lands on the intended cluster, ShortCircuit runs **0.61–0.65** against Linus Tech Tips'
> **0.87–1.04**. Both figures are relative to each channel's own recent neighbours, so this is not
> channel size restated.
>
> The three points drawn as open circles are configurations where the anchor terms pulled in an
> adjacent cluster instead — a laptops cluster once, a pure water-cooling cluster once, an
> unscripted cluster once. They are reported rather than dropped, and they point the same way,
> which is weak corroboration and not additional evidence.
>
> **What this is worth.** ShortCircuit spends roughly 8% of its post-decline output on the subject
> it measurably handles worst, and which the sister channel handles at baseline. It is the only
> place in this project where public data supports a specific operational suggestion instead of a
> description. It is also narrow: it says this subject underperforms *on this channel*, not that
> the videos are bad, and not that moving them would recover the reach.
>
> **What it is not.** An explanation. Why the same subject works on one channel and not the other
> needs click-through and retention by traffic source — owner-only, as everywhere else here.

> ## ⚠️ 2026-07-30 · 复核与修正：这一节的检验不是撑钱的那个检验
>
> 上面的 24 组配置比的是 **ShortCircuit vs 旗舰**，同一题材。那回答的是"这个题材在别处能不能做
> 成",不是"这个题材在本频道低于本频道自己多少"——而后者才是把它换算成播放量和钱的那一步所需要的。
> 用同一套 24 组配置重跑了本频道内的版本，另外把口径推到最保守的地方：
>
> | 检验 | 同向 | p<0.05 | 中位比 |
> |---|---|---|---|
> | SC vs 旗舰（本节已发表） | 24/24 | 24/24 | — |
> | **SC 装机 vs SC 其余视频** | 24/24 | 24/24 | 0.621 |
> | **同上，同龄口径 + 用播放量而非 rbi** | 24/24 | 23/24 | 0.622 |
>
> 同龄口径下的中位播放是 **248,400 对 395,697**。所以这条发现既不是 rbi 的假象，也不是只在跨频道
> 成立——它在钱所用的那个单位上活了下来。
>
> **但由它换算出的金额是错的，错了约三倍。** `GOAL.md` 情形 B 写的是「挪到中位约 +770 万播放 ≈
> +$0.19M/**年**」。770 万 = 49 支 × 每支缺口，而这 49 支摊在**整整三年**的后下滑窗口里，却被
> 摆在一个年度数字（语速那条）旁边相加。按年重算：
>
> ```
> 15.4 支/年 × 147,297 次播放/支 = 226 万播放/年 → $5.7 万/年 @ $25 CPM
> ```
>
> 演示里的天花板因此从 **+35% 降到 +29%**。这和 Shorts 那次是同一类错误：**两个不同时间基的量
> 相加**。写下任何一个"合计"之前，先问两个加数的分母是不是同一个。
>
> 复核脚本按 `ncomp` 分三次跑（沙箱里后台进程活不过一次调用），中间产物 `st_100/150/200.csv`。
> 结论已同步进 `GOAL.md` 3c、`SCRIPT.md` §5、deck slide 10 及其敏感度表。

### Where the storefront does and does not line up

The bridge to anything commercial is deliberately weak, and the weakness is the finding.

The store endpoint returns products, variants, prices and stock status. It does **not** return
units sold, so there is no way to say a subject converts. What can be said is narrower: which
subjects the company currently earns attention with, and whether the catalogue has anything to sell
against them. A subject that performs with no product behind it is an unpriced option; a category
with many products and no matching content is an exposure liability.

In [ ]:
store = pd.read_csv(DATA/"lttstore_skus_2026-07-26.csv")
prod  = store.drop_duplicates("handle")

print(f"{len(prod)} products · {len(store)} variants · "
      f"{1 - store.available.mean():.1%} of variants out of stock\n")
cat = (store.groupby("product_type")
            .agg(products=("handle", "nunique"), variants=("sku", "size"),
                 oos=("available", lambda s: 1 - s.mean()))
            .sort_values("products", ascending=False))
print(cat.assign(oos=(cat.oos*100).round(1)).to_string())

In [ ]:
# Keyword match between a subject's top terms and product titles. Crude on purpose:
# a fuzzy match would invent precision this data cannot support.
titles = prod["product"].str.lower()
rows = []
for c in range(14):
    s = (post.cluster == c).values
    if s.sum() < 30:
        continue
    top = terms[np.asarray(X[s].mean(axis=0)).ravel().argsort()[::-1][:15]]
    kws = [t for t in top if len(t) > 3 and " " not in t]
    rows.append({"subject": label[c], "videos": int(s.sum()),
                 "rbi": round(post.loc[s, "rbi"].median(), 2),
                 "products matching": int(titles.apply(lambda t: any(k in t for k in kws)).sum())})
gap = pd.DataFrame(rows).sort_values("rbi", ascending=False)
print(gap.to_string(index=False))

print("\nRead the right-hand column as catalogue coverage, not as demand. Zero products")
print("against a high-rbi subject means there is nothing to sell there — not that")
print("something would sell if there were. And apparel, the largest category by variants,")
print("matches no subject at all, which is the point made in 03.")

### Remotion plate — the stability scatter, not the company-wide panel

The presentation uses the **right** panel of the figure above, not the left one. The left panel is a
company-wide view of thirteen subjects, and the talk is about one channel; showing it would spend
attention establishing something the argument does not use.

The right panel is the one that belongs: it is ShortCircuit against the flagship on a single subject,
across 24 parameter configurations, and every point sitting below the diagonal is the credibility
beat.

Reads `topic_stability_post.csv` rather than recomputing, so this cell is cheap to re-run.

In [ ]:
REM = FIG/"remotion"; REM.mkdir(exist_ok=True)
stab_csv = pd.read_csv(DATA/"topic_stability_post.csv")
stab_csv["on_target"] = stab_csv.top.str.contains("power supply|motherboard|fans")

def stability_plate(dark, out_png, out_json=None, figsize=(9.6, 5.4), dpi=150):
    INK  = "#E8ECF2" if dark else "#1B2430"
    GRID = "#39414E" if dark else "#C9D2DE"
    MUTE = "#8A94A6" if dark else "#6B7684"
    ON, OFF = ("#4DA3FF", "#FF6B5E") if dark else ("#2E86DE", "#E74C3C")
    plt.rcParams.update({"text.color": INK, "axes.labelcolor": INK, "xtick.color": INK,
                         "ytick.color": INK, "axes.edgecolor": GRID, "grid.color": GRID,
                         "font.size": 13})
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.plot([.5, 1.3], [.5, 1.3], ls="--", color=MUTE, lw=1.2, zorder=1)
    ax.set_xlim(.55, 1.28); ax.set_ylim(.55, 1.28)
    ax.set_xlabel("Linus Tech Tips — median rolling-baseline index")
    ax.set_ylabel("ShortCircuit — median rolling-baseline index")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=.28, lw=.8)
    ax.text(.97, .12, "Every point below the diagonal means ShortCircuit\n"
                      "underperforms the flagship on the same subject.",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=12, color=MUTE, linespacing=1.5)
    from matplotlib.lines import Line2D
    on_n, off_n = int(stab_csv.on_target.sum()), int((~stab_csv.on_target).sum())
    leg = ax.legend(handles=[
        Line2D([], [], marker="o", ls="none", color=ON, ms=9,
               label=f"anchor on the build-hardware cluster ({on_n})"),
        Line2D([], [], marker="o", ls="none", markerfacecolor="none", markeredgecolor=OFF,
               markeredgewidth=1.8, ms=9, label=f"anchor matched another cluster ({off_n})")],
        frameon=False, fontsize=12, loc="upper left")
    for t in leg.get_texts(): t.set_color(INK)
    plt.tight_layout()
    fig.savefig(out_png, bbox_inches=None, transparent=dark)
    series = [{"name": "on target", "color": ON, "kind": "scatter", "filled": True,
               "x": [float(v) for v in stab_csv.LTT[stab_csv.on_target]],
               "y": [float(v) for v in stab_csv.SC[stab_csv.on_target]]},
              {"name": "off target", "color": OFF, "kind": "scatter", "filled": False,
               "x": [float(v) for v in stab_csv.LTT[~stab_csv.on_target]],
               "y": [float(v) for v in stab_csv.SC[~stab_csv.on_target]]}]
    if out_json:
        w, h = fig.get_size_inches()*fig.dpi; pos = ax.get_position()
        json.dump({"figure_px": {"w": w, "h": h}, "dpi": fig.dpi,
                   "axes": [{"px": {"x": pos.x0*w, "y": (1-pos.y1)*h,
                                    "w": pos.width*w, "h": pos.height*h},
                             "xlim": list(ax.get_xlim()), "ylim": list(ax.get_ylim()),
                             "xscale": "linear", "yscale": "linear"}],
                   "series": series,
                   "summary": {"configs": int(len(stab_csv)),
                               "sc_below_ltt": int((stab_csv.SC < stab_csv.LTT).sum()),
                               "p_under_05": int((stab_csv.p < .05).sum()),
                               "on_target": on_n}}, open(out_json, "w"), indent=1)
    plt.show(); plt.close(fig); plt.rcParams.update(plt.rcParamsDefault)

stability_plate(True,  REM/"stability_plate.png", REM/"stability.json")
stability_plate(False, FIG/"stability.png", dpi=110)
print(f"{len(stab_csv)} configurations · "
      f"ShortCircuit below the flagship in {(stab_csv.SC < stab_csv.LTT).sum()} · "
      f"p < 0.05 in {(stab_csv.p < .05).sum()}")

## 3b. Attachment rate — the one commercial quantity that is actually public

> ⚠️⚠️ **Two findings in this section were killed by `08`. Read `08` before citing anything here.**
>
> - **The Floatplane "single-month cliff" was a URL migration.** They moved outbound links onto
>   their own shortener, `lmg.gg`, in February 2023. Any mention of Floatplane runs 99.1% of
>   uploads in 2021 and **100% in 2026** — it never left. The chart in this section was measuring
>   a string rewrite.
> - **The affiliate decline was regex under-coverage.** Under a taxonomy built from a census of
>   every URL in the corpus, commercial outbound links run **100% in 2021 and 100% in 2026**.
>
> What survives: long-form storefront attachment at 99% (flat), and the Shorts result — which `08`
> confirms and sharpens.
>
> Left in place rather than rewritten, because the error and its correction are worth more together.



> ⚠️ **Corrected in `07`, section 5. Read that before using anything below.**
>
> The headline in this section — blended attachment falling from 98.2% to 86.1%, decomposed into
> "reach −29% × attachment −12%" — is **wrong**. Splitting by format shows long-form attachment
> never moved (99.1% → 99.5%); the blended rate fell only because Shorts, which never carried
> links on more than ~58% of uploads, grew from 17 uploads a year to 90. It is a composition
> effect read as a behaviour change — the same mistake as the AI cluster in `05` and the
> 45-minute filter in `02`.
>
> What survives: the Floatplane break (long-form, 89.5% → 1.7% in one month) and the affiliate
> decline (96.8% → 36.5%) are real and are not composition. The genuine storefront gap is Shorts:
> 205M views in 2025, 26% of the network, 2.2% carrying a link.
>
> The section is left in place rather than rewritten, because the error and its correction are
> more useful together than a clean version would be.

Everything up to here measures attention. The obvious next question is what the attention is
attached to, and the honest answer for most of this project has been "nothing observable" — the
storefront endpoint carries no units sold.

There is one exception, and it was sitting in data collected back in `04`. **The description is a
monetisation surface.** Whether a video carries a link to the storefront is a decision the company
makes per upload, it is fully public, and it has never been looked at here.

In creator-economy practice this is the middle tier of a standard three-tier framework: business
outcomes (attributed revenue, sales lift, CAC) sit on top, campaign performance (click-through on
creator-tagged traffic, click-to-purchase) in the middle, content health (reach, engagement rate,
completion) underneath. **Tier one is invisible to us and tier three is what the rest of this
notebook measures. Attachment rate is a foothold in tier two** — not conversion, but the exposure
that conversion would have to run through.

In [ ]:
desc = read_desc() if "read_desc" in dir() else pd.DataFrame(
    [json.loads(l) for l in (DATA/"content/descriptions.jsonl").open() if l.strip()]
).drop_duplicates("id", keep="last")

SURFACE = {"store": r"lttstore\.com",
           "floatplane": r"floatplane\.com",
           "affiliate": r"amzn\.to|geni\.us|howl\.link|tag="}
for k, pat in SURFACE.items():
    desc[k] = desc.description.str.contains(pat, case=False, na=False)

mon = desc.merge(vids[["id", "channel", "date", "views"]], on="id")
mon["yr"] = mon.date.dt.year

print("Attachment rate by year, four channels combined\n")
print((mon[mon.yr.between(2021, 2025)].groupby("yr")[list(SURFACE)].mean()*100).round(1).to_string())

print("\n\nStore attachment by channel and year\n")
print((mon[mon.yr.between(2021, 2026)]
       .pivot_table(index="yr", columns="channel", values="store", aggfunc="mean")*100).round(1).to_string())

print("\n\nThe Floatplane break, monthly\n")
mm = mon[mon.date.between("2022-10-01", "2023-06-30")].copy()
mm["ym"] = mm.date.dt.to_period("M")
print((mm.groupby("ym")[["floatplane", "store", "affiliate"]].mean()*100).round(1).to_string())

### Multiplying the two declines

Reach and attachment rate are independent decisions and they multiply. Neither number alone
describes what happened to the storefront's exposure.

In [ ]:
expo = (mon[mon.yr.between(2021, 2025)].groupby("yr")
        .apply(lambda g: pd.Series({
            "uploads":   len(g),
            "attach":    g.store.mean(),
            "total_M":   g.views.sum()/1e6,
            "exposed_M": g.loc[g.store, "views"].sum()/1e6}), include_groups=False))
print(expo.round(3).to_string())
expo.to_csv(DATA/"store_exposure.csv")

b, e = expo.loc[2021], expo.loc[2025]
print(f"\n2021 → 2025")
print(f"  uploads          {b.uploads:>6.0f} → {e.uploads:<6.0f} {e.uploads/b.uploads-1:>+7.0%}")
print(f"  attachment rate  {b.attach:>6.1%} → {e.attach:<6.1%} {e.attach/b.attach-1:>+7.0%}")
print(f"  all views        {b.total_M:>5.0f}M → {e.total_M:<5.0f}M {e.total_M/b.total_M-1:>+7.0%}")
print(f"  store exposure   {b.exposed_M:>5.0f}M → {e.exposed_M:<5.0f}M {e.exposed_M/b.exposed_M-1:>+7.0%}")
print("\nThe two declines compound. Reporting reach alone understates what happened")
print("to the storefront by roughly a third.")

fig, ax = plt.subplots(1, 2, figsize=(13.6, 4.8), gridspec_kw={"width_ratios": [1.25, 1]})
mon["ym"] = mon.date.dt.to_period("M").dt.to_timestamp()
q = mon[mon.date >= "2021-01-01"].groupby("ym")[list(SURFACE)].mean()*100
for c, col, lab in [("store", "#2E86DE", "Own storefront (lttstore.com)"),
                    ("affiliate", "#F39C12", "Affiliate links"),
                    ("floatplane", "#E74C3C", "Floatplane")]:
    ax[0].plot(q.index, q[c], color=col, lw=1.9, label=lab)
ax[0].axvline(pd.Timestamp("2023-02-01"), color="#888", ls=":", lw=1.2)
ax[0].annotate("Feb 2023\n82.8% → 1.4% in one month", xy=(pd.Timestamp("2023-02-01"), 46),
               xytext=(pd.Timestamp("2023-05-01"), 62), fontsize=8.3, color="#E74C3C",
               arrowprops=dict(arrowstyle="->", color="#E74C3C", lw=.9))
ax[0].set_ylim(-4, 108); ax[0].set_ylabel("Share of uploads carrying the link (%)")
ax[0].set_title("Three monetisation surfaces, all withdrawing", loc="left", weight="bold")
ax[0].legend(frameon=False, fontsize=8.5, loc="center left")
ax[0].text(.985, .72, "Monthly, four channels combined.\nThe description is a monetisation\n"
                      "surface, and it is being vacated.",
           transform=ax[0].transAxes, ha="right", va="top", fontsize=8.2, color="#555")

x = np.arange(len(expo))
ax[1].bar(x, expo.total_M, color="#D6E4F0", label="all views")
ax[1].bar(x, expo.exposed_M, color="#2E86DE", label="views on uploads carrying a store link")
for i, val in enumerate(expo.exposed_M):
    ax[1].text(i, val-62, f"{val:.0f}M", ha="center", color="white", fontsize=8.5, weight="bold")
ax[1].set_xticks(x); ax[1].set_xticklabels(expo.index); ax[1].set_ylabel("Views (millions)")
ax[1].set_ylim(0, 1720)
ax[1].set_title("Store exposure fell 45%, not 29%", loc="left", weight="bold")
ax[1].legend(frameon=False, fontsize=8.3, loc="upper right")
ax[1].text(.03, .30, "Reach              −29%\nAttachment rate    −12%\n" + "─"*24 +
                     "\nStore exposure     −45%",
           transform=ax[1].transAxes, fontsize=8.6, color="#333", family="monospace", va="top")
for a in ax: a.spines[["top", "right"]].set_visible(False); a.grid(axis="y", alpha=.22)
plt.tight_layout(); plt.savefig(FIG/"monetisation_surface.png", bbox_inches="tight"); plt.show()

> **What this says.** Between 2021 and 2025 reach fell 29% and the share of uploads carrying a
> storefront link fell from 98.2% to 86.1%. The two are separate decisions and they multiply:
> **views on uploads that actually carry a store link fell 45%**, from 1,069M to 592M. Reporting
> the reach decline alone understates the storefront's lost exposure by about a third.
>
> **A dated break.** Floatplane links sat above 90% of uploads through 2022 and then went from
> **82.8% in January 2023 to 1.4% in February 2023** — one month, and never recovered. Affiliate
> links had already fallen from ~96% to ~45% during the first half of 2022, and drift down to 32%
> by 2025. Three monetisation surfaces, all being vacated, on three different schedules.
>
> **This closes a question left open in `03`.** That notebook could not rule out the reading that
> falling YouTube reach was partly deliberate — value being moved behind the company's own
> paywall. If that were the strategy, Floatplane promotion would have intensified. Instead it was
> removed outright in a single month. The reading is now much harder to sustain. *Why* it was
> removed is not in this data, and I am not going to guess.
>
> **What it is not.** Attachment rate is exposure, not conversion. A link in a description is not
> a click and a click is not a sale. Click-through on those links is owner-only, like everything
> else that would settle a commercial question here. What can be said is that the surface through
> which any storefront revenue would have to arrive got 45% smaller, and that roughly a quarter of
> that shrinkage was a choice rather than an algorithm.

## 4. Next steps

Written as next steps because that is what they are. None is established by the work above, and
each names what would have to be true.

**1 — Test the ShortCircuit build-hardware result against something other than clustering.**
It survived 24 configurations, which rules out the partition being the cause. It does not rule out
the subject being a proxy for something else: format, length, whether the video is a sponsored
build, whether it went out in a week when the flagship covered the same product. A regression on
the video attributes already in `videos_clean.csv`, with the subject as one term among several,
would say whether the subject is carrying the effect or riding on it.

**2 — Per channel, never company-wide.** Two findings in this project now say the same thing. The
apparent company-wide pivot to AI content was one channel covering the news. The company-wide
subject table spans only 0.86–1.15 and looks like nothing, while inside it one channel-subject
pair sits at 0.61. Aggregating across channels destroyed both. Any subject-level recommendation
has to be made per channel or it will be pointed the wrong way for someone.

**3 — Techquickie needs a different instrument.** Its median transcript is 6,361 characters
against the flagship's 16,784, so fixed presenter language is a much larger share of the text and
the channel resists subject analysis at this resolution. Titles plus descriptions, or a
sentence-level model applied to the middle of each transcript, would probably do better. Nothing
in this notebook covers Techquickie well and that should not be papered over.

**4 — A daily storefront snapshot.** Stock status is observable and moves; a product going out of
stock and returning is a weak but real signal about turnover. One snapshot exists, taken
2026-07-26, so no series exists. Starting now yields something usable in weeks and nothing for
this deadline — which is exactly why it belongs here rather than in the results.

**5 — The measurement that would settle the main question.** Separating content from distribution
needs click-through rate, average view duration and retention. All three are owner-only. This is
not a gap more public collection can close, and it is worth stating plainly rather than working
around: everything above describes what reached people, never why.